# Mu-Logarit (91 dạng, 8.190 biến thể) - FULL90 - ZERO-SHOT - Qwen3-4B

**Dùng chung 1 notebook cho cả 10 tài khoản Kaggle.** Trước khi "Run All", chỉ cần đổi `GROUP_ID` (1-10) ở ô cấu hình bên dưới cho đúng tài khoản đang chạy - không cần sửa gì khác.

Prompt, tham số sinh (`temperature=0.6, top_p=0.95, top_k=20, seed=42, presence_penalty=1.2, max_new_tokens=9216, enable_thinking=True`) và cách chấm điểm giữ nguyên y hệt bản Qwen3-4B zero-shot đã dùng cho bộ Số phục.

**CẦN UPLOAD** `plan_solve_prompts_mulogarit_merged.json` lên Kaggle làm 1 Dataset rồi "Add Data" vào notebook (tên dataset tuỳ ý) - notebook tự dò đường dẫn thật dưới `/kaggle/input/` bằng `glob`, không cần gõ cứng đường dẫn.


In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec torchaudio
import vllm; print('vLLM:', vllm.__version__)


## Cấu hình - ĐỔI `GROUP_ID` CHO ĐÚNG TÀI KHOẢN (1-10), KHÔNG SỬA GÌ KHÁC Ở Ô NÀY

In [ ]:
import os, json, re, time, glob
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

GROUP_ID = 1  # <-- SUA SO NAY (1-10) THEO TUNG TAI KHOAN, roi Run All

DATA_FILENAME = 'plan_solve_prompts_mulogarit_merged.json'
# Duong dan /kaggle/input/<ten-dataset>/... khac nhau tuy ten dataset ban
# dat khi upload; thay vi go cung duong dan, tu do tim file nay o bat ky
# dataset nao da duoc gan (Add Data) vao notebook.
_matches = glob.glob(f'/kaggle/input/**/{DATA_FILENAME}', recursive=True)
assert _matches, (f'Khong tim thay {DATA_FILENAME} duoi /kaggle/input/ - '
                  'kiem tra da "Add Data" dataset chua chuoi file nay chua.')
MERGED_DATA_PATH = _matches[0]
print('Dung file du lieu:', MERGED_DATA_PATH)

# Danh sach ma dang theo tung group (1-10), nhung 10 tai khoan deu dung
# chung notebook nay - chi GROUP_ID doi.
GROUPS = {
    "1": [
        "D01",
        "D02",
        "D04",
        "D05",
        "D06",
        "D07",
        "D08",
        "D10",
        "D12"
    ],
    "2": [
        "D13",
        "D14",
        "D15",
        "D16",
        "D17",
        "D18",
        "D19",
        "D20",
        "D21"
    ],
    "3": [
        "D22",
        "D23",
        "D24",
        "D25",
        "D27",
        "D28",
        "D29",
        "D30",
        "D33"
    ],
    "4": [
        "D34",
        "D35",
        "D37",
        "D38",
        "D39",
        "D40",
        "D41",
        "D42",
        "D43"
    ],
    "5": [
        "D44",
        "D45",
        "D46",
        "D47",
        "D48",
        "D49",
        "D51",
        "D52",
        "D53"
    ],
    "6": [
        "D54",
        "D55",
        "D56",
        "D57",
        "D58",
        "D59",
        "D60",
        "D61",
        "D62"
    ],
    "7": [
        "D63",
        "D64",
        "D65",
        "D66",
        "D67",
        "D68",
        "D69",
        "D71",
        "D72"
    ],
    "8": [
        "D73",
        "D74",
        "D75",
        "D76",
        "D77",
        "D78",
        "D79",
        "D80",
        "D81"
    ],
    "9": [
        "D82",
        "D83",
        "D84",
        "D85",
        "D86",
        "D87",
        "D88",
        "D89",
        "D90"
    ],
    "10": [
        "D91",
        "D92",
        "D93",
        "D94",
        "D36a",
        "D36b",
        "D50a",
        "D50b"
    ]
}

TYPES = GROUPS[str(GROUP_ID)]
MODEL = 'Qwen/Qwen3-4B'
MAX_MODEL_LEN, MAX_NEW_TOKENS = 11264, 9216
TEMPERATURE, TOP_P, TOP_K, SEED = 0.6, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(MERGED_DATA_PATH, encoding='utf-8') as f:
    ALL_RECORDS = json.load(f)

print(f'GROUP_ID={GROUP_ID}: {len(TYPES)} dang, {sum(len(ALL_RECORDS[t]) for t in TYPES)} cau')
print('Danh sach dang:', TYPES)


## Nạp model (1 lần cho cả nhóm dạng)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')
sampling = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                          presence_penalty=PRESENCE_PENALTY,
                          max_tokens=MAX_NEW_TOKENS, seed=SEED)


## Sinh lời giải + chấm điểm cho từng dạng trong nhóm

In [ ]:
PROMPT_TEMPLATE = r'''Bạn hãy giải bài toán trắc nghiệm sau đây.

Yêu cầu trình bày:
- Giải chi tiết từng bước bằng tiếng Việt.
- Viết dưới dạng Markdown, dùng LaTeX cho công thức toán (đặt công thức trong dấu $...$).
- Không xuống dòng quá nhiều lần trong một biểu thức (ví dụ ngoặc vuông, phân số) vì sẽ khó đọc.
- Sau khi giải xong, kết luận rõ ràng ở dòng cuối cùng: "Đáp án đúng: X" (X là A, B, C hoặc D).

Đề bài:
{de_bai}
'''


In [ ]:
def get_letter(t):
    m = re.findall(r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b', t)
    if m:
        return m[-1]
    m = re.findall(r'\\boxed\{([A-D])\}', t)
    return m[-1] if m else ''

summary = []
for ma_dang in TYPES:
    records = ALL_RECORDS[ma_dang]
    out_path = f'/kaggle/working/{ma_dang.lower()}_zeroshot_full90_Qwen3-4B.csv'

    prompts = []
    for r in records:
        txt = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
        prompts.append(tok.apply_chat_template(
            [{'role': 'user', 'content': txt}],
            tokenize=False, add_generation_prompt=True, enable_thinking=True))

    t0 = time.time()
    outputs = llm.generate(prompts, sampling)
    print(f'{ma_dang}: sinh xong {len(outputs)} cau, mat {(time.time()-t0)/60:.1f} phut.')

    rows = []
    for r, out in zip(records, outputs):
        txt = out.outputs[0].text
        ntok = len(out.outputs[0].token_ids)
        think = txt.split('</think>')[0] if '</think>' in txt else txt
        after = txt.split('</think>')[-1] if '</think>' in txt else ''
        het_ngan_sach = ntok >= MAX_NEW_TOKENS - 1
        chon = get_letter(after)
        rows.append({
            'STT': r['STT'], 'ma_dang': r['ma_dang'],
            'dap_an_dung': r['dap_an_letter'], 'model_chon': chon,
            'DUNG': chon == r['dap_an_letter'],
            'token': ntok, 'het_ngan_sach': het_ngan_sach,
            'think': think, 'after': after,
        })

    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    summary.append({'ma_dang': ma_dang, 'dung': int(df['DUNG'].sum()), 'tong': len(df)})
    print(f"  -> {ma_dang}: {df['DUNG'].sum()}/{len(df)} dung ({df['DUNG'].mean()*100:.1f}%)")


## Tổng kết nhóm dạng vừa chạy

In [ ]:
df_summary = pd.DataFrame(summary)
print('='*70)
print(f'KET QUA NHOM GROUP_ID={GROUP_ID} ({len(TYPES)} dang):')
print(df_summary)
print('-'*70)
print(f"TONG: {df_summary['dung'].sum()}/{df_summary['tong'].sum()} dung "
      f"({df_summary['dung'].sum()/df_summary['tong'].sum()*100:.1f}%)")
print('='*70)
